# Practice: the grounding reward

The reward turns one episode into a number in [0, 1]: parse the final reply, keep only cites the model actually retrieved, check each claim against its cited evidence, measure coverage of the gold points, subtract a penalty for unsupported claims and a precision penalty for supported claims that aren't gold, and handle abstention by the item's `answerable` label. `docs/reward_spec.md` defines it.

Reward v1.1 (set 2026-11-20, the default): v0's constants (λ = 0.5, κ = 0.2, abstaining on a correctable item scores 0.7, flag-only answers count on partial items) plus the **precision penalty** μ · max(0, N − a) / S, with μ = 1 and an allowance a = 3. N counts supported facts that state no gold point, and S the supported distinct facts. Reward v1 (`V1_PARAMS`) divided the same term by F, all distinct facts; that let fabricated claims dilute it (audit candidate 12). Reward v0 is the formula with μ = 0 (`V0_PARAMS`). With no unsupported facts, S = F and v1 and v1.1 agree. Three rules sit on top: **strict citations** (one cite that wasn't retrieved voids the whole claim), **duplicate facts count once** (claims stating the same fact collapse before S and U are counted), and the final answer is the **last sampled assistant message** (the harness's `root_reply` is ignored).

This notebook builds small traces with the shared test builders, scores them with the real reward, and then scores them again with a **copy** of the aggregate defined here, so you can change λ, the partial-flag credit κ, the precision weight μ, the allowance a, the precision term's denominator (`over="supported"` for v1.1, `over="facts"` for v1), or the formula itself and watch scores move. It never edits `src/`: it calls the public `neuro_grounded_qa` API and imports the builders from `tests/`.

Everything is synthetic: the fixture store has invented IDs like `GENE_A`, `DIS_1`, `E:000001`.

In [ ]:
import json
import sys
from pathlib import Path

# The builders live in tests/; put tests/ on the path, as pytest does.
REPO = Path.cwd() if (Path.cwd() / "tests").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "tests"))

from contracts.builders import TraceBuilder, call, edge_payload, fixture_store
from neuro_grounded_qa.contracts import Abstain, Answer, QuestionItem, parse_output, retrieved_evidence
from neuro_grounded_qa.reward import (
    V0_PARAMS, V1_PARAMS, FactGroup, RewardParams, check_claims, coverage, directed_relations, evaluate, final_reply, flagged,
    gold_fact, group_facts, precision_penalty, score, unsupported_penalty,
)

STORE = fixture_store()
ITEMS = {
    item.id: item
    for item in (
        QuestionItem.model_validate_json(line)
        for line in (REPO / "tests/fixtures/contracts/items.jsonl").read_text().splitlines()
        if line.strip()
    )
}
SINGLE, PARTIAL, UNANS = ITEMS["fx-single-001"], ITEMS["fx-partial-001"], ITEMS["fx-unans-001"]
print(SINGLE.prompt_text)
for gp in SINGLE.gold_points:
    print(" gold:", gp.subject_id, gp.relation, gp.object_id, gp.evidence)

## 1. Building episodes

`episode(item, final, retrieved)` makes a one-branch trace: the prompt, one sampled `get_edge` call per retrieved edge (each answered by a real tool result), then the model's final reply. `claim(...)` and `answer(...)` write the final reply as the model would.

In [ ]:
def claim(s, r, o, *cites):
    return {"subject_id": s, "relation": r, "object_id": o, "cites": list(cites)}

def answer(*claims, unsupported=()):
    body = {"claims": list(claims), "answer_text": "see claims"}
    if unsupported:
        body["unsupported_parts"] = list(unsupported)
    return json.dumps(body)

ABSTAIN = json.dumps({"abstain": True, "reason": "the evidence doesn't support an answer"})

def episode(item, final, retrieved=()):
    b = TraceBuilder(data=item)
    b.system("Answer only from retrieved evidence.")
    b.user(item.prompt_text)
    for i, eid in enumerate(retrieved):
        b.assistant(None, [call(f"call_{i}", "get_edge", edge_id=eid)])
        b.tool(f"call_{i}", edge_payload(eid), name="get_edge")
    b.assistant(final)
    return b.build()

GP1 = claim("GENE_A", "associated_with", "DIS_1", "E:000001")
GP2 = claim("GENE_B", "associated_with", "DIS_1", "E:000002")

## 2. Four answers to the single-hop question

- **honest**: both gold points, plus one honest mistake (cites a real edge that says something else).
- **padded**: the honest answer plus four true, retrieved, irrelevant claims (one more than the allowance).
- **fabricated**: cites `E:000002` without ever retrieving it.
- **abstain**: declines.

Plus the **evidence dump**: every edge touching `DIS_1` restated as a claim.

In [ ]:
MISTAKE = claim("GENE_C", "associated_with", "DIS_1", "E:000009")  # E:000009 is GENE_C - DIS_2
PADDING = [
    claim("GENE_A", "member_of", "PATH_P", "E:000007"),
    claim("GENE_C", "member_of", "PATH_P", "E:000008"),
    claim("GENE_C", "associated_with", "DIS_2", "E:000009"),
    claim("GENE_A", "interacts_with", "GENE_C", "E:000011"),
]
DIS_1_EDGES = STORE.neighbors("DIS_1", relation=None, limit=50)

EPISODES = {
    "honest": episode(SINGLE, answer(GP1, GP2, MISTAKE), ["E:000001", "E:000002", "E:000009"]),
    "padded": episode(
        SINGLE, answer(GP1, GP2, MISTAKE, *PADDING),
        ["E:000001", "E:000002", "E:000007", "E:000008", "E:000009", "E:000011"],
    ),
    "fabricated": episode(SINGLE, answer(GP1, GP2), ["E:000001"]),
    # Both gold edges retrieved, but the second claim also cites an ID never fetched.
    "one bad cite": episode(
        SINGLE,
        answer(GP1, claim("GENE_B", "associated_with", "DIS_1", "E:000002", "E:999999")),
        ["E:000001", "E:000002"],
    ),
    # The honest answer with its supported claims repeated (once reversed) as padding.
    "repeats": episode(
        SINGLE,
        answer(GP1, GP2, MISTAKE, GP1, GP2, claim("DIS_1", "associated_with", "GENE_A", "E:000001")),
        ["E:000001", "E:000002", "E:000009"],
    ),
    "abstain": episode(SINGLE, ABSTAIN, ["E:000001"]),
    "evidence dump": episode(
        SINGLE,
        answer(*[claim(e.subject.id, e.relation, e.object.id, e.id) for e in DIS_1_EDGES]),
        [e.id for e in DIS_1_EDGES],
    ),
}

def row(name, b):
    cov = "n/a" if b.coverage is None else f"{b.coverage:.2f}"
    return (f"{name:14s} reward={b.reward:.3f}  cov={cov}  S={b.n_supported} U={b.n_unsupported}"
            f" N={b.n_nongold_supported}  dup={b.n_duplicates} voided={b.n_voided}"
            f"  penalty={b.penalty:.3f} precision={b.precision_penalty:.3f}  outcome={b.outcome}")

for name, trace in EPISODES.items():
    print(row(name, evaluate(trace, SINGLE).breakdown))
print()
print("Under reward v1 (precision term over all facts F):")
print(row("padded", evaluate(EPISODES["padded"], SINGLE, V1_PARAMS).breakdown))
print()
print("Under reward v0 (no precision penalty):")
for name in ("padded", "evidence dump"):
    print(row(name, evaluate(EPISODES[name], SINGLE, V0_PARAMS).breakdown))

Things to notice:

- The fabricated cite is stripped: the claim it backed is unsupported, so the answer gets half coverage and pays a penalty.
- **One bad cite voids the claim.** In "one bad cite" the GENE_B claim cites the real, retrieved `E:000002` *and* the never-fetched `E:999999`. Under strict citations the claim is unsupported (`voided=1`), so it scores like "fabricated".
- **Repeats don't dilute.** "repeats" restates its two supported claims (once with subject and object swapped). They collapse into the same two facts (`dup=3`), so S and U, and the score, match "honest".
- **Padding past the allowance costs.** Under v0, four true, supported, irrelevant facts diluted the mistake's penalty (λ·1/7 instead of λ·1/3), so "padded" beat "honest". Under v1.1, the fourth non-gold fact is one past the allowance and costs μ/S = 1/6 (S = 6 supported facts; the mistake is unsupported), so "padded" falls below "honest". Under v1 it cost μ/F = 1/7: the mistake counted in the denominator. Up to three would still be free.
- **The fixture's evidence dump still scores 1.0.** `DIS_1` has only four neighbors, two of them gold, so the dump states N = 2 non-gold facts: within the allowance. On the real graph a dump states about 55 facts per answer, and v1 and v1.1 take it to nearly 0 (`docs/reward_hacking_audit.md`, part 3).

## 3. A copy of the aggregate to experiment on

`my_aggregate` below is a copy of `neuro_grounded_qa.reward.scoring.aggregate`, and `my_score` wires it to the real components, including fact deduplication (`group_facts`) and the precision penalty (`gold_fact`, `precision_penalty`). Its defaults are reward v1.1: pass `over="facts"` for v1, `mu=0` for v0. Edit either one here; `src/` is untouched.

In [ ]:
def my_aggregate(answerable, outcome, cov, penalty, is_flagged, n_claims,
                 lam=0.5, kappa=0.2, correctable_abstain=0.7, flag_only_counts=True):
    if outcome == "parse_failure":
        return 0.0
    abstained = outcome == "abstain"
    if answerable == "yes":
        return 0.0 if abstained else (cov or 0.0) - penalty
    if answerable == "partial":
        if abstained:
            return 0.0
        flag = 1.0 if is_flagged and (n_claims > 0 or flag_only_counts) else 0.0
        return (1 - kappa) * (cov or 0.0) + kappa * flag - penalty
    if answerable == "no":
        return 1.0 if abstained else 0.0
    if answerable == "correctable":
        return correctable_abstain if abstained else (cov or 0.0) - penalty
    raise ValueError(answerable)


def my_score(item, trace, aggregate=my_aggregate, dedupe=True, mu=1.0, allowance=3, over="supported", **params):
    parsed = parse_output(final_reply(trace))  # the last sampled message, not root_reply
    retrieved = retrieved_evidence(trace)
    if isinstance(parsed, Answer):
        checks = check_claims(parsed, retrieved, item)
        cov, _ = coverage(item, checks)
        is_flagged = flagged(parsed)
        outcome = "answer"
    else:
        checks, cov, is_flagged = (), (0.0 if item.gold_points else None), False
        outcome = "abstain" if isinstance(parsed, Abstain) else "parse_failure"
    # S and U count distinct facts; dedupe=False counts claims as written instead.
    # Claims carry no direction flag: directed relations come from the retrieved edges
    # and the item, so a reversed hierarchy claim stays its own fact.
    units = group_facts(checks, directed_relations(item, retrieved)) if dedupe else checks
    n_sup = sum(u.supported for u in units)
    penalty = unsupported_penalty(n_sup, len(units) - n_sup, params.get("lam", 0.5))
    # Precision penalty: supported facts that state no gold point, past the allowance,
    # over the supported facts (v1.1, over="supported") or over every fact (v1, over="facts").
    # With dedupe=False each claim is its own fact.
    facts = units if dedupe else [FactGroup((c,)) for c in checks]
    n_nongold = sum(f.supported and not gold_fact(f, item) for f in facts)
    denominator = sum(f.supported for f in facts) if over == "supported" else len(facts)
    penalty += precision_penalty(n_nongold, denominator, mu, allowance)
    raw = aggregate(item.answerable, outcome, cov, penalty, is_flagged, len(checks), **params)
    return min(1.0, max(0.0, raw))


# The copy agrees with the real reward at the default constants (v1.1), at v1's and at v0's.
for name, trace in EPISODES.items():
    assert abs(my_score(SINGLE, trace) - evaluate(trace, SINGLE).breakdown.reward) < 1e-12, name
    assert abs(my_score(SINGLE, trace, over="facts") - evaluate(trace, SINGLE, V1_PARAMS).breakdown.reward) < 1e-12, name
    assert abs(my_score(SINGLE, trace, mu=0) - evaluate(trace, SINGLE, V0_PARAMS).breakdown.reward) < 1e-12, name
print("copy matches the package")

## 4. Sweep λ

Watch the honest answer fall as λ rises, while the padded answer falls more slowly (dilution) and the evidence dump doesn't move at all: its claims are all supported, so only the precision penalty can reach it.

In [ ]:
lams = [0.0, 0.25, 0.5, 1.0, 2.0]
print(f"{'':14s}" + "".join(f"λ={lam:<6}" for lam in lams))
for name, trace in EPISODES.items():
    print(f"{name:14s}" + "".join(f"{my_score(SINGLE, trace, lam=lam):<8.3f}" for lam in lams))

## 5. Sweep μ and the allowance a

The honest answer plus *k* true, supported claims that aren't gold (``context''), for k = 0 … 9: the fixture store's edges that the single-hop item doesn't ask about (all but the mistake's edge). Under v1.1 (μ = 1, a = 3) the first three are free; each one after that costs μ/S. Every claim here is supported, so S = F and v1 gives the same numbers. Try other allowances and weights, including μ = 0 (reward v0).

In [ ]:
NONGOLD = [claim(e.subject.id, e.relation, e.object.id, e.id) for e in (STORE.get(f"E:{i:06d}") for i in range(3, 13))
           if e.id not in ("E:000009",)]  # E:000009 is GENE_C - DIS_2: kept out so it isn't confused with MISTAKE
CONTEXT = {k: episode(SINGLE, answer(GP1, GP2, *NONGOLD[:k]), ["E:000001", "E:000002", *(c["cites"][0] for c in NONGOLD[:k])])
           for k in range(0, len(NONGOLD) + 1)}

settings = [(0.0, 3), (0.5, 3), (1.0, 0), (1.0, 2), (1.0, 3), (1.0, 4)]
print(f"{'k':>3s}" + "".join(f"  μ={mu:g},a={a}" for mu, a in settings))
for k, trace in CONTEXT.items():
    print(f"{k:>3d}" + "".join(f"  {my_score(SINGLE, trace, mu=mu, allowance=a):>9.3f}" for mu, a in settings))

## 6. Unsupported claims and the precision penalty (candidate 12)

Under v1, F counted every distinct fact, unsupported ones included. So when an answer's total penalty was above λ, adding fabricated claims *lowered* it: the precision share shrank faster than the unsupported share grew, and the total tended to λ. v1.1 divides by S, which a fabricated claim doesn't change, so fabrications only add to the unsupported penalty. Take the context answer with k = 9 (six non-gold facts past the allowance) and add *u* guesses: v1.1 falls, v1 rises, v0 only falls.

In [ ]:
def guesses(u):
    # Each guess cites a retrieved edge that doesn't state it: unsupported, not voided.
    return [claim(f"GENE_FAKE{i}", "associated_with", "DIS_1", "E:000001") for i in range(u)]

k = len(NONGOLD)
cited = ["E:000001", "E:000002", *(c["cites"][0] for c in NONGOLD)]
print(f"{'u':>4s}{'v1.1':>8s}{'v1':>8s}{'v0':>8s}")
for u in (0, 2, 5, 10, 25, 50):
    t = episode(SINGLE, answer(GP1, GP2, *NONGOLD, *guesses(u)), cited)
    print(f"{u:>4d}{my_score(SINGLE, t):>8.3f}{my_score(SINGLE, t, over='facts'):>8.3f}{my_score(SINGLE, t, mu=0):>8.3f}")

## 7. Sweep κ on the partial item

The partial item asks two things; only the first is in the graph. Compare a complete answer with and without the flag, a half answer with the flag, a flag-only answer and an abstention.

In [ ]:
FLAG = ["no drug acting on Gene Delta is in the graph"]
PARTIAL_EPISODES = {
    "full + flag": episode(PARTIAL, answer(GP1, GP2, unsupported=FLAG), ["E:000001", "E:000002"]),
    "full, no flag": episode(PARTIAL, answer(GP1, GP2), ["E:000001", "E:000002"]),
    "half + flag": episode(PARTIAL, answer(GP1, unsupported=FLAG), ["E:000001"]),
    "flag only": episode(PARTIAL, answer(unsupported=FLAG)),
    "abstain": episode(PARTIAL, ABSTAIN),
}
kappas = [0.0, 0.1, 0.2, 0.4]
print(f"{'':14s}" + "".join(f"κ={k:<6}" for k in kappas))
for name, trace in PARTIAL_EPISODES.items():
    print(f"{name:14s}" + "".join(f"{my_score(PARTIAL, trace, kappa=k):<8.3f}" for k in kappas))

## 8. Swap the formula: the brief's additive partial credit

The brief's wording reads as `coverage + κ·flag − penalty`, clipped. Under that form, a complete answer scores 1 with or without the flag. Compare it with the weighted form the package uses.

In [ ]:
def additive_aggregate(answerable, outcome, cov, penalty, is_flagged, n_claims, kappa=0.2, **kw):
    if answerable == "partial" and outcome == "answer":
        return (cov or 0.0) + kappa * float(is_flagged) - penalty
    return my_aggregate(answerable, outcome, cov, penalty, is_flagged, n_claims, kappa=kappa, **kw)

print(f"{'':14s}{'weighted':>10s}{'additive':>10s}")
for name, trace in PARTIAL_EPISODES.items():
    w = my_score(PARTIAL, trace)
    a = my_score(PARTIAL, trace, aggregate=additive_aggregate)
    print(f"{name:14s}{w:>10.3f}{a:>10.3f}")

## 9. Always-abstain over the target type mix

The brief's target mix is single-hop 30%, multi-hop 25%, partial 15%, unanswerable 20%, false premise 10% (split evenly here between the `no` and `correctable` fixture items). Change the correctable-abstain credit and see how the floor moves: 0.7 is reward v0 (0.285); 1.0 was the first proposal (0.30).

In [ ]:
MIX = {"fx-single-001": 0.30, "fx-multi-001": 0.25, "fx-partial-001": 0.15,
       "fx-unans-001": 0.20, "fx-falsep-001": 0.05, "fx-falsep-002": 0.05}

for credit in (1.0, 0.7, 0.5):
    total = sum(w * my_score(ITEMS[i], episode(ITEMS[i], ABSTAIN), correctable_abstain=credit)
                for i, w in MIX.items())
    print(f"correctable abstain credit {credit}: always-abstain scores {total:.3f}")

## Things to try

- Candidate 12 is closed in v1.1 by the denominator. Try the other fix that was on the table, an absolute charge (μ · max(0, N − a), capped, with no denominator): what does it do to the context sweep in section 5 and to the guesses in section 6?
- Make the allowance depend on the question: a share of the gold points instead of a fixed 3.
- Score "repeats" with `my_score(..., dedupe=False)` and watch the duplicates dilute the penalty again.
- Make flags cost something on `yes` items.
- Retrieve the fixture's directed edge `E:000013` (`DIS_3 subtype_of DIS_G`) and claim it both ways round: the reversal is unsupported and is not absorbed as a duplicate.